In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约1-2分钟）
!pip install -q numpy scipy matplotlib soundfile torch torchaudio
!apt-get install -qq ffmpeg sox
print('✅ 环境配置完成！')

# 调试、版本控制与编程习惯

## 学习目标

- 掌握基本的调试方法（print → pdb → Jupyter调试器）
- 学会读懂常见Python报错信息
- 掌握Git基本操作（init, add, commit, push, pull）
- 建立良好的编程习惯（命名规范、注释、格式化）


## 1. 调试方法

调试是编程能力中最被忽视的，也是提升编程自信最有效的。

物理背景学生遇到报错时最常见的反应是"代码报错了"然后等待救援，而不是自己读报错信息、定位问题。

**目标**：建立"报错是朋友不是敌人"的心态——报错信息就是在告诉你哪里出了问题，学会读它就能解决大部分问题。

### 1.1 最常见的报错类型

在深度学习/音频处理中，最常遇到的报错：

In [ ]:
# ====== 故意制造报错，学习读报错 ======

# 1. IndexError: 索引越界
lst = [1, 2, 3]
try:
    lst[5]
except IndexError as e:
    print(f'IndexError: {e}')

In [ ]:
# 2. KeyError: 字典键不存在
d = {'a': 1, 'b': 2}
try:
    d['c']
except KeyError as e:
    print(f'KeyError: {e}')

In [ ]:
# 3. ValueError: 值错误
try:
    int('hello')
except ValueError as e:
    print(f'ValueError: {e}')

In [ ]:
# 4. TypeError: 类型错误
try:
    '2' + 2
except TypeError as e:
    print(f'TypeError: {e}')

In [ ]:
# 5. RuntimeError: 运行时错误（PyTorch中最常见）
# 最典型的：维度不匹配
import torch
import torch.nn as nn

linear = nn.Linear(10, 5)  # 期望输入维度为10
x = torch.randn(3, 8)      # 实际输入维度为8
try:
    y = linear(x)
except RuntimeError as e:
    print(f'RuntimeError: {e}')
    print()
    print('关键信息解读:')
    print('  - Mat1和Mat2维度不匹配')
    print('  - 最后一维: 8 vs 10')
    print('  - 意思是: 输入x的最后一个维度是8，但线性层期望10')
    print('  - 解决: 检查x.shape，确认最后一维是否正确')

### 报错信息阅读技巧

1. **从下往上读**——最后一行是错误类型和消息，最有信息量
2. **看Traceback**——它告诉你错误发生在哪一行、调用链是什么
3. **读最后一行两遍**——第一遍理解大意，第二遍找关键数字
4. **维度不匹配是最常见的错误**——养成习惯：每步都 `print(tensor.shape)`

### 1.2 调试方法演进

| 方法 | 优点 | 缺点 |
|------|------|------|
| print | 最简单 | 需要手动添加/删除 |
| pdb | 不需要改代码 | 交互式命令行，不太直观 |
| Jupyter调试器 | 可视化断点 | 只在notebook中有效 |
| VS Code调试器 | 最专业 | 需要配置 |

**建议**：在日常开发中，`print(tensor.shape)` 仍然是最快最实用的调试方法。

In [ ]:
# print调试示例：跟踪张量形状变化
import torch
import torch.nn as nn

class DebugCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)  # 输入1通道，输出16通道
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)  # 16→32
        self.pool = nn.MaxPool2d(2, 2)  # 尺寸减半
        self.fc = nn.Linear(32 * 8 * 8, 10)  # 全连接层
    
    def forward(self, x):
        # 每一步都打印形状
        print(f'输入: {x.shape}')
        
        x = self.pool(torch.relu(self.conv1(x)))
        print(f'conv1 + pool: {x.shape}')
        
        x = self.pool(torch.relu(self.conv2(x)))
        print(f'conv2 + pool: {x.shape}')
        
        x = x.view(x.size(0), -1)  # 展平
        print(f'flatten: {x.shape}')
        
        x = self.fc(x)
        print(f'fc输出: {x.shape}')
        return x

# 测试
model = DebugCNN()
x = torch.randn(4, 1, 32, 32)  # batch=4, channel=1, 32x32
y = model(x)

In [ ]:
# pdb 调试示例：Python 内置调试器
# pdb 不需要修改原代码——在怀疑出错的位置插入断点即可

import pdb

def buggy_function(data, threshold):
    """这个函数在某个条件下会出错——用 pdb 找到原因"""
    total = 0
    for i, x in enumerate(data):
        # 如果想在这里设断点，取消下一行注释
        # pdb.set_trace()  # 程序会停在这里，进入交互式调试
        if x > threshold:
            total += x * 2
        else:
            total -= x
    return total

# 演示 pdb 基本命令（在终端运行 python 后才会真的进入 pdb）
# 在终端运行：
#   python -m pdb script.py        # 从头调试整个脚本
#   或在代码里加 pdb.set_trace()   # 在指定位置停下

# pdb 常用命令：
# n (next)        — 执行下一行（不进入函数）
# s (step)        — 执行下一行（进入函数）
# c (continue)    — 继续执行到下一个断点
# p <var>         — 打印变量值（如 p x, p total）
# l (list)        — 显示当前代码附近
# w (where)       — 显示调用栈
# q (quit)        — 退出调试
# b <line>        — 在指定行设断点（如 b 15）
# 空回车           — 重复上一条命令

# 注意：在 Jupyter notebook 中 pdb.set_trace() 会显示在输出区，不如终端方便
# Jupyter 有更好的调试方式（见下一个 cell）

# 这里只演示概念，不真的进入 pdb（会卡住 notebook）
print("pdb 命令参考已就绪")
print("在终端运行: python -m pdb your_script.py  进入 pdb 调试")

### Jupyter 内置调试器（推荐在 notebook 中用）

Jupyter 提供两个魔法命令，比 pdb 更友好：

| 命令 | 用途 | 何时用 |
|------|------|--------|
| `%debug` | **事后调试**——在报错发生后回溯栈，逐帧检查变量 | 出错后想查"刚才变量值是什么" |
| `%%debug` | 整个 cell 进入调试模式 | 预计会出错，提前准备 |
| `set_trace()`（IPython） | 主动设断点 | 想停在指定位置 |

**`%debug` 演示流程**（在 notebook 中试）：
1. 先运行一个会报错的 cell
2. 在新 cell 中只输入 `%debug` 并运行
3. 进入交互式调试器（ipdb），可用 `u`/`d` 上下栈、`p var` 打印、`q` 退出

```python
# 在报错 cell 之后，新建一个 cell 只写：
# %debug
# 就能进入事后调试——回看每个变量在出错前的值
```

**主动断点**（推荐，比 pdb.set_trace() 在 Jupyter 中更友好）：
```python
from IPython.core.debugger import set_trace

def my_func(x):
    set_trace()    # 程序停在这里，弹出交互式调试器
    return x * 2
```

> **VS Code 调试器** 是最专业的选择——可以可视化断点、变量监视、调用栈。在 VS Code 中打开 `.py` 文件，点击行号左侧设断点，按 F5 启动调试。对于 `.ipynb` 文件，VS Code 也支持 cell 级别的断点。

## 2. Git基础

Git是版本管理工具，在研究中它的作用是：

- **保存进度**：每次实验结果有改进就commit，随时可以回退
- **同步代码**：在服务器和笔记本之间同步
- **记录实验**：commit message就是实验记录


In [ ]:
# Git基本操作（在终端中执行，这里只展示命令）

# ===== 1. 本地仓库基本操作 =====

# 初始化仓库（在当前目录创建.git）
# git init

# 添加文件到暂存区
# git add .                  # 添加所有改动
# git add notebook.ipynb     # 只添加某个文件

# 提交（相当于保存进度）
# git commit -m "完成模块0的练习"

# 查看当前状态
# git status

# 查看历史
# git log --oneline

# 回退到之前的版本
# git checkout <commit-hash>

# ===== 2. 远程仓库操作（push/pull/clone） =====

# 克隆远程仓库到本地（第一次获取代码用）
# git clone https://github.com/username/repo.git
# git clone git@github.com:username/repo.git    # SSH方式（推荐）

# 查看远程仓库地址
# git remote -v

# 添加远程仓库（已有本地仓库，要推送到远程）
# git remote add origin git@github.com:username/repo.git

# 推送本地提交到远程
# git push origin master       # 第一次推送
# git push                     # 之后简写

# 拉取远程更新到本地（其他人推送了新代码）
# git pull

# ===== 3. 推送本地文件夹到 GitHub 的完整流程 =====
# 场景：你在本地写了一个项目，想推送到 GitHub

# step 1: 在 GitHub 网页上创建一个空仓库（不要勾选 README/gitignore）
# step 2: 在本地项目目录执行：
# cd /path/to/your/project
# git init
# git add .
# git commit -m "initial commit"
# git branch -M main              # GitHub 默认分支名是 main
# git remote add origin git@github.com:username/repo.git
# git push -u origin main         # -u 设置上游，之后可以 git push 简写

# ===== 4. 协作流程（多人开发） =====
# 每次开始工作前：git pull        # 同步最新代码
# 写代码 → git add → git commit
# 推送：git push
# 如果 push 失败（远程有新提交）：先 git pull，解决冲突，再 git push

# ===== 5. 配置（首次使用 Git 必做） =====
# git config --global user.name "Your Name"
# git config --global user.email "your@email.com"

print("Git命令参考已就绪")
print("完整流程：clone → edit → add → commit → pull → push")

### 远程仓库与协作流程

**远程仓库**：放在服务器上的 Git 仓库（GitHub/Gitee/学校 GitLab）。本地仓库与远程仓库通过 `push`/`pull` 同步。

#### 三种典型场景

**场景 1：从零开始，已有本地代码 → 推送到 GitHub**

```bash
# 在 GitHub 网页创建空仓库（不要勾选 README）
cd /path/to/your/project
git init
git add .
git commit -m "initial commit"
git branch -M main
git remote add origin git@github.com:username/repo.git
git push -u origin main
```

**场景 2：已有远程仓库 → 克隆到本地**

```bash
git clone git@github.com:username/repo.git
cd repo
# 修改代码...
git add .
git commit -m "修改说明"
git push
```

**场景 3：团队协作——别人推送了新代码**

```bash
# 开始工作前
git pull                    # 拉取最新代码
# 修改代码...
git add .
git commit -m "我的修改"
git pull                    # 再次拉取，避免冲突
git push                    # 推送
```

#### 为什么 `git push` 会失败？

最常见的错误：`! [rejected] main -> main (fetch first)`

原因：远程仓库有你本地没有的提交（别人推送了）。解决：
```bash
git pull                    # 先合并远程的改动
# 如果有冲突，编辑冲突文件，删除 <<<<<<< ======= >>>>>>> 标记
git add .                   # 标记冲突已解决
git commit -m "merge"
git push                    # 再推送
```

#### SSH vs HTTPS

| 方式 | 命令 | 优点 | 缺点 |
|------|------|------|------|
| SSH | `git@github.com:user/repo.git` | 不用每次输密码 | 需配置 SSH key |
| HTTPS | `https://github.com/user/repo.git` | 简单 | 每次要输密码/token |

实验室服务器推荐 SSH（配一次 key，之后无感）。

#### 培训期间的实际使用

```bash
# 第一次获取课程代码
git clone git@github.com:lab/lab-training.git
cd lab-training

# 每次课前更新
git pull

# 课后提交练习
git add .
git commit -m "module0: 完成Python基础练习"
git push
```

> **重要**：培训期间你的 commit message 就是实验记录。好的 message 一个月后还能看懂当时做了什么。


### Git commit message 规范

好的commit message就是实验记录：

```bash
# 好的例子
git commit -m "CNN分类器：2层conv+1层fc，准确率85%"
git commit -m "修改学习率：0.001→0.0001，loss下降更快"
git commit -m "增加数据增强：随机裁剪+频谱掩蔽，过拟合缓解"

# 坏的例子
git commit -m "update"
git commit -m "fix bug"
git commit -m "修改了一下"
```

**原则**：commit message要回答"这次改动做了什么，效果如何？"

## 3. 编程习惯

### 3.1 命名规范

| 类型 | 规范 | 例子 |
|------|------|------|
| 变量/函数 | snake_case | `sample_rate`, `compute_spectrum()` |
| 类 | PascalCase | `AudioDataset`, `CNNClassifier` |
| 常量 | UPPER_CASE | `SAMPLE_RATE`, `NUM_CLASSES` |
| 私有属性 | _前缀 | `self._internal_state` |

### 3.2 好名字 vs 坏名字

**变量名要回答"这个变量存储什么"，而不是"这个变量的类型是什么"。**

In [ ]:
# ====== 坏名字 vs 好名字 ======

# 坏名字
a = 16000          # 什么采样率？
b = 0.5            # 什么的振幅？
c = np.zeros(100)  # 什么的缓冲区？
tmp = wave[:500]   # 什么的临时变量？

# 好名字
sample_rate = 16000
amplitude = 0.5
output_buffer = np.zeros(100)
first_500_samples = wave[:500]

### 3.3 注释写法

**注释要解释"为什么"，而不是"做了什么"**——代码本身已经说明了"做了什么"。

In [ ]:
# ====== 坏注释 vs 好注释 ======

# 坏注释：重复代码
sr = 16000  # 设置采样率为16000
n = int(sr * dur)  # 计算样本数

# 好注释：解释"为什么"
sample_rate = 16000  # CI语音处理的常用采样率
num_samples = int(sample_rate * duration)  # int()截断而非四舍五入，避免最后一个样本溢出

### 3.4 代码格式化

使用 `black` 自动格式化代码，不再为格式争论浪费时间。

In [ ]:
# 安装black（在终端中执行）
# pip install black

# 使用black格式化当前notebook
# black notebook.ipynb
# 或者格式化整个项目
# black .

print('代码格式化工具介绍完毕')

### 3.5 类型提示 (Type Hints)

类型提示让 Python（动态语言）也能有"静态类型检查"——IDE 能自动补全、运行前能发现类型错误。

**类型提示不影响运行**（Python 解释器不检查），但让代码更易读、易维护，IDE（VS Code、PyCharm）会帮你检查。

#### 基本语法

```python
# 没有类型提示
def add(a, b):
    return a + b

# 有类型提示
def add(a: int, b: int) -> int:
    return a + b
```

#### 常用类型

| 类型 | 含义 | 例子 |
|------|------|------|
| `int`, `float`, `str`, `bool` | 基本类型 | `x: int = 1` |
| `list[int]` | int 列表 | `xs: list[int] = [1, 2, 3]` |
| `dict[str, int]` | 字典 | `d: dict[str, int] = {"a": 1}` |
| `tuple[int, str]` | 固定长度元组 | `t: tuple[int, str] = (1, "a")` |
| `Optional[int]` | 可选（int 或 None） | `x: Optional[int] = None` |
| `Union[int, str]` | 联合类型 | `x: Union[int, str] = 1` |
| `np.ndarray` | NumPy 数组 | 需 `import numpy as np` |
| `torch.Tensor` | PyTorch 张量 | 需 `import torch` |

#### 何时用类型提示？

- **公开函数**（被其他模块调用）：必须加——这是契约
- **私有辅助函数**：建议加——便于自己回头理解
- **notebook 里的临时代码**：可加可不加
- **快速原型/实验脚本**：不必加——别让类型提示拖慢探索速度

> **建议**：写研究代码时，先把模型和数据流跑通（不加类型），跑通后再加类型提示做"清理"——这能帮你发现隐含的假设（如"我以为 x 是 4D tensor，实际是 3D"）。
>
> **工具**：`mypy your_script.py` 做静态类型检查；VS Code 装 Python 扩展自动检查。

In [ ]:
# ====== 类型提示示例 ======
from typing import Optional, Union, List, Dict
import numpy as np
import torch


# 示例 1: 基本类型提示
def compute_snr(signal: np.ndarray, noise: np.ndarray) -> float:
    """计算信噪比 (dB)。"""
    sig_power = np.mean(signal ** 2)
    noise_power = np.mean(noise ** 2)
    return 10 * np.log10(sig_power / (noise_power + 1e-12))


# 示例 2: 容器类型（Python 3.9+ 可直接用 list/dict，旧版本需 List/Dict）
def normalize_batch(spectrograms: list[np.ndarray], eps: float = 1e-8) -> list[np.ndarray]:
    """归一化一批语谱图。"""
    return [(s - s.mean()) / (s.std() + eps) for s in spectrograms]


# 示例 3: Optional（参数可空）
def load_audio(path: str, target_sr: Optional[int] = None) -> tuple[np.ndarray, int]:
    """加载音频，target_sr 为 None 时不重采样。"""
    audio, sr = ...  # 省略实现
    if target_sr is not None and target_sr != sr:
        # 重采样
        pass
    return audio, sr


# 示例 4: 模型类（深度学习最常见）
class AudioClassifier(torch.nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64) -> None:
        super().__init__()
        self.n_classes: int = n_classes
        self.n_mels: int = n_mels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x 期望形状: [batch, n_mels, time]
        # 返回形状: [batch, n_classes]
        return x  # 省略实现


# 类型提示不影响运行——下面这行不会报错（但 mypy 会警告）
def buggy(x: int) -> int:
    return "这不是 int"  # 运行时不报错，但类型检查器会标红

# IDE 会提示 buggy 的返回类型与声明不符——这就是类型提示的价值
print("类型提示示例已就绪")
print("提示: 在 VS Code 中打开此文件，悬停在函数签名上能看到类型提示")

## 4. 综合练习

**任务**：将前两次课的代码重构为模块化的项目，提交到Git仓库。

### 步骤

1. 创建项目目录结构：
   ```
   audio_tools/
   ├── __init__.py
   ├── signal.py          # Signal类
   ├── generators.py      # SineWave, NoisySignal
   └── visualization.py   # 画图工具函数
   ```
2. 将notebook中的代码整理到对应的.py文件中
3. 在notebook中用import引用自己的模块
4. 用Git提交，commit message写清楚


In [ ]:
# ====== 综合练习：模块化重构 ======

# 步骤1：创建目录结构
import os

project_dir = 'audio_tools'
os.makedirs(project_dir, exist_ok=True)

# 创建__init__.py（让Python识别这个目录为包）
with open(os.path.join(project_dir, '__init__.py'), 'w') as f:
    f.write('')

print(f'项目目录 {project_dir} 创建完成')

In [ ]:
# 步骤2：将Signal类写入signal.py
signal_code = '''
import numpy as np
import matplotlib.pyplot as plt

class Signal:
    """音频信号基类"""
    
    def __init__(self, waveform, sample_rate, label=''):
        self.waveform = waveform
        self.sample_rate = sample_rate
        self.label = label
    
    def plot(self):
        sr = self.sample_rate
        t = np.linspace(0, len(self.waveform)/sr, len(self.waveform), endpoint=False)
        plt.figure(figsize=(12, 4))
        plt.plot(t, self.waveform)
        plt.title(self.label)
        plt.show()
    
    def compute_spectrum(self):
        N = len(self.waveform)
        spectrum = np.abs(np.fft.rfft(self.waveform))
        freqs = np.fft.rfftfreq(N, 1/self.sample_rate)
        return freqs, spectrum
'''

with open(os.path.join(project_dir, 'signal.py'), 'w') as f:
    f.write(signal_code)
print('signal.py 写入完成')

In [ ]:
# 步骤2b：将SineWave和NoisySignal写入generators.py
generators_code = '''
import numpy as np
from .signal import Signal

class SineWave(Signal):
    """正弦波信号"""
    
    def __init__(self, frequency, amplitude=0.5, duration=1.0, sample_rate=16000):
        t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
        waveform = amplitude * np.sin(2 * np.pi * frequency * t)
        label = f'{frequency} Hz Sine Wave'
        super().__init__(waveform, sample_rate, label)
        self.frequency = frequency
        self.amplitude = amplitude
        self.duration = duration

class NoisySignal(Signal):
    """带噪信号"""
    
    def __init__(self, clean_waveform, sample_rate, noise_level=0.1, label=''):
        noise = np.random.randn(len(clean_waveform)) * noise_level
        noisy = clean_waveform + noise
        label = label or f'Noisy Signal (noise_level={noise_level})'
        super().__init__(noisy, sample_rate, label)
        self.clean_waveform = clean_waveform
        self.noise_level = noise_level
    
    def compute_snr(self):
        signal_power = np.mean(self.clean_waveform ** 2)
        noise = self.waveform - self.clean_waveform
        noise_power = np.mean(noise ** 2)
        if noise_power == 0:
            return float('inf')
        return 10 * np.log10(signal_power / noise_power)
'''

with open(os.path.join(project_dir, 'generators.py'), 'w') as f:
    f.write(generators_code)
print('generators.py 写入完成')

In [ ]:
# 步骤3：测试从模块导入
from audio_tools.signal import Signal
from audio_tools.generators import SineWave, NoisySignal

# 使用方式和之前完全一样
sine = SineWave(440)
print(f'频率: {sine.frequency} Hz')
print(f'采样率: {sine.sample_rate} Hz')

clean = SineWave(440)
noisy = NoisySignal(clean.waveform, clean.sample_rate, noise_level=0.3)
print(f'SNR: {noisy.compute_snr():.1f} dB')

## 本节要点

1. **读报错信息**：从下往上读，最后一行最有信息量；维度不匹配是最常见的bug
2. **调试方法**：`print(tensor.shape)` 是最快最实用的；Jupyter调试器和VS Code调试器更专业
3. **Git**：`init → add → commit` 是最基本的工作流；commit message要写清楚
4. **编程习惯**：好名字回答"存储什么"；注释解释"为什么"；用black自动格式化


---
返回目录：[README.md](../README.md)